# 2026-03 OCR Pipeline Comparison

End-to-end comparison of three PDF text-extraction approaches:
- **PyMuPDF** — native PDF parsing (no OCR, uses embedded text layer)
- **Tesseract** — OCR engine on rasterized page images
- **Docling** — AI-based document understanding (layout + table detection)

Each method is run as an intensive pipeline over all files. Results are collected into a unified DataFrame for direct comparison.

- Example and fake data used for this study:
    - Microsoft Azure invoices: https://www.microsoft.com/en-us/download/details.aspx?id=38805
    - Kaggle receipts (2024-EN): https://www.kaggle.com/datasets/jenswalter/receipts

**Dependencies**: `uv pip install pymupdf pytesseract pillow docling`

---

### Setup
Importing all necessary libs and getting a list with all the example data

In [ ]:
# Install dependencies (run once)
# !uv pip install pymupdf pytesseract pillow docling

In [ ]:
# Python imports
import os, sys, time, json
from pathlib import Path

# Data handling imports
import pandas as pd
import numpy as np
from tqdm import tqdm
from PIL import Image

# --> PyMuPDF
import pymupdf

# --> Tesseract
import pytesseract
from pytesseract import Output

# --> Docling
from docling.document_converter import DocumentConverter
from docling.datamodel.base_models import ConversionStatus
from docling.chunking import HybridChunker

In [ ]:
# Path to all the example files
path = Path('./data/')

# Get all files
files_list = [f'./data/{entry.name}' for entry in path.rglob('*') if entry.is_file()]

# Sort the list
files_list = sorted(files_list)

print(f"Found {len(files_list)} files:")
for f in files_list:
    print(f"  {f}")

### Helper functions
One self-contained pipeline function per method — each returns a unified result dict

In [ ]:
def run_pymupdf_pipeline(file_path: str) -> dict:
    """
    Intensive PyMuPDF pipeline.
    Per page: plain text, blocks, words, dict extraction, image scan, table detection.
    Returns a result dict with extracted text and structural metadata.
    """
    t0 = time.perf_counter()

    doc = pymupdf.open(file_path)

    all_text, all_blocks, all_words, all_images, all_tables = [], [], [], [], []

    for page_num in range(doc.page_count):
        page = doc.load_page(page_num)

        # All text extraction modes
        all_text.append(page.get_text("text"))
        blocks = page.get_text("blocks")
        all_blocks.extend(blocks)
        all_words.extend(page.get_text("words"))

        # Embedded images
        all_images.extend(page.get_images(full=True))

        # Table detection
        found = page.find_tables()
        all_tables.extend(found.tables)

    full_text = "\n".join(all_text)
    text_blocks = [b for b in all_blocks if b[6] == 0]  # type 0 = text block

    return {
        'file': Path(file_path).name,
        'method': 'pymupdf',
        'pages': doc.page_count,
        'word_count': len(full_text.split()),
        'char_count': len(full_text),
        'block_count': len(text_blocks),
        'word_bbox_count': len(all_words),
        'table_count': len(all_tables),
        'image_count': len(all_images),
        'mean_confidence': None,  # not applicable for native text
        'time_s': round(time.perf_counter() - t0, 3),
        'text': full_text,
        'tables': [t.extract() for t in all_tables],
    }

In [ ]:
def run_tesseract_pipeline(file_path: str, zoom: float = 3.0) -> dict:
    """
    Intensive Tesseract pipeline.
    Renders each PDF page at zoom=3.0 (~216 DPI) for best OCR accuracy.
    Per page: image_to_string (psm=3, oem=3), image_to_data, image_to_boxes.
    Returns a result dict with extracted text and confidence statistics.
    """
    t0 = time.perf_counter()

    doc = pymupdf.open(file_path)
    mat = pymupdf.Matrix(zoom, zoom)

    all_text, all_conf, all_words = [], [], []

    for page_num in range(doc.page_count):
        page = doc.load_page(page_num)
        pix = page.get_pixmap(matrix=mat)
        img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)

        # Best-accuracy config: LSTM engine, auto page segmentation
        cfg = "--psm 3 --oem 3"

        # Plain text
        text = pytesseract.image_to_string(img, config=cfg)
        all_text.append(text)

        # Word-level data with confidence scores
        df_data = pytesseract.image_to_data(img, config=cfg, output_type=Output.DATAFRAME)
        conf_words = df_data[df_data['conf'] > 0]
        all_conf.extend(conf_words['conf'].tolist())
        all_words.extend(conf_words['text'].dropna().tolist())

    full_text = "\n".join(all_text)
    mean_conf = round(float(np.mean(all_conf)), 1) if all_conf else 0.0
    high_conf = sum(1 for c in all_conf if c >= 80)

    return {
        'file': Path(file_path).name,
        'method': 'tesseract',
        'pages': doc.page_count,
        'word_count': len(full_text.split()),
        'char_count': len(full_text),
        'block_count': None,
        'word_bbox_count': len(all_words),
        'table_count': None,  # tesseract does not detect tables
        'image_count': None,
        'mean_confidence': mean_conf,
        'high_conf_words': high_conf,
        'time_s': round(time.perf_counter() - t0, 3),
        'text': full_text,
        'tables': [],
    }

In [ ]:
def run_docling_pipeline(file_path: str, converter: DocumentConverter) -> dict:
    """
    Intensive Docling pipeline.
    Converts the document, exports markdown + dict, extracts tables as DataFrames,
    and chunks the document with HybridChunker.
    Returns a result dict with extracted text and structural metadata.
    """
    t0 = time.perf_counter()

    result = converter.convert(file_path)
    doc = result.document

    # Full markdown export
    md = doc.export_to_markdown()

    # Tables as DataFrames
    tables_dfs = []
    for table in doc.tables:
        try:
            tables_dfs.append(table.export_to_dataframe())
        except Exception:
            pass

    # Chunking for RAG
    chunker = HybridChunker()
    chunks = list(chunker.chunk(doc))

    return {
        'file': Path(file_path).name,
        'method': 'docling',
        'pages': None,
        'word_count': len(md.split()),
        'char_count': len(md),
        'block_count': len(doc.texts),
        'word_bbox_count': None,
        'table_count': len(doc.tables),
        'image_count': len(doc.pictures),
        'mean_confidence': None,
        'chunk_count': len(chunks),
        'status': str(result.status),
        'time_s': round(time.perf_counter() - t0, 3),
        'text': md,
        'tables': tables_dfs,
    }

### Pipeline: PyMuPDF
Native PDF text layer — fastest method, no AI, no OCR

In [ ]:
# Run PyMuPDF pipeline over all files
pymupdf_results = []

for file_path in tqdm(files_list, desc="PyMuPDF"):
    res = run_pymupdf_pipeline(file_path)
    pymupdf_results.append(res)
    print(f"  {res['file']:50s} | words={res['word_count']:5d} | tables={res['table_count']} | images={res['image_count']} | {res['time_s']}s")

In [ ]:
# Summary DataFrame — PyMuPDF
df_pymupdf = pd.DataFrame([
    {k: v for k, v in r.items() if k not in ('text', 'tables')}
    for r in pymupdf_results
])
display(df_pymupdf)

In [ ]:
# Text preview — invoice (files_list[0])
ref = pymupdf_results[0]
print(f"=== PyMuPDF text preview: {ref['file']} ===")
print(ref['text'][:800])

### Pipeline: Tesseract
OCR on rasterized pages — works even when the PDF has no text layer (scanned documents)

In [ ]:
# Run Tesseract pipeline over all files (zoom=3.0 → ~216 DPI for best accuracy)
tesseract_results = []

for file_path in tqdm(files_list, desc="Tesseract"):
    res = run_tesseract_pipeline(file_path, zoom=3.0)
    tesseract_results.append(res)
    print(f"  {res['file']:50s} | words={res['word_count']:5d} | conf={res['mean_confidence']} | {res['time_s']}s")

In [ ]:
# Summary DataFrame — Tesseract
df_tesseract = pd.DataFrame([
    {k: v for k, v in r.items() if k not in ('text', 'tables')}
    for r in tesseract_results
])
display(df_tesseract)

In [ ]:
# Text preview — invoice (files_list[0])
ref = tesseract_results[0]
print(f"=== Tesseract text preview: {ref['file']} ===")
print(ref['text'][:800])

### Pipeline: Docling
AI-based document understanding — richest structure, table detection, chunking

In [ ]:
# Initialize converter once — reuse across all files to avoid reloading models
converter = DocumentConverter()

In [ ]:
# Run Docling pipeline over all files
docling_results = []

for file_path in tqdm(files_list, desc="Docling"):
    res = run_docling_pipeline(file_path, converter)
    docling_results.append(res)
    print(f"  {res['file']:50s} | words={res['word_count']:5d} | tables={res['table_count']} | chunks={res['chunk_count']} | {res['time_s']}s")

In [ ]:
# Summary DataFrame — Docling
df_docling = pd.DataFrame([
    {k: v for k, v in r.items() if k not in ('text', 'tables')}
    for r in docling_results
])
display(df_docling)

In [ ]:
# Text preview — invoice (files_list[0])
ref = docling_results[0]
print(f"=== Docling text preview: {ref['file']} ===")
print(ref['text'][:800])

In [ ]:
# Extracted tables from the invoice
ref = docling_results[0]
print(f"=== Docling tables: {ref['file']} — {len(ref['tables'])} table(s) ===\n")
for i, df_t in enumerate(ref['tables']):
    print(f"Table {i}:")
    display(df_t)
    print()

### Comparison
Side-by-side metrics across all three methods

In [ ]:
# Build unified comparison DataFrame — one row per (file, method)
COLS = ['file', 'method', 'word_count', 'char_count', 'table_count', 'image_count', 'mean_confidence', 'time_s']

rows = []
for group in [pymupdf_results, tesseract_results, docling_results]:
    for r in group:
        rows.append({c: r.get(c) for c in COLS})

df_comparison = pd.DataFrame(rows)

# Pivot so each file gets one row with columns per method
df_pivot = df_comparison.pivot(index='file', columns='method', values=['word_count', 'char_count', 'table_count', 'mean_confidence', 'time_s'])
df_pivot.columns = ['_'.join(col) for col in df_pivot.columns]
df_pivot = df_pivot.reset_index()

print("=== Full comparison (pivoted) ===")
display(df_pivot)

In [ ]:
# Aggregate stats — mean per method across all files
df_agg = (
    df_comparison
    .groupby('method')[['word_count', 'char_count', 'time_s', 'mean_confidence']]
    .agg(['mean', 'min', 'max'])
    .round(2)
)
print("=== Aggregate stats per method ===")
display(df_agg)

In [ ]:
# Side-by-side text preview — same reference file (invoice) from all three methods
PREVIEW_LEN = 400
ref_file = Path(files_list[0]).name

def get_text(results, filename):
    for r in results:
        if r['file'] == filename:
            return r['text'][:PREVIEW_LEN]
    return ''

print(f"Reference file: {ref_file}")
print()

for method, results in [("PyMuPDF", pymupdf_results), ("Tesseract", tesseract_results), ("Docling", docling_results)]:
    text = get_text(results, ref_file)
    print("="*80)
    print(f"[{method}]")
    print("="*80)
    print(text)
    print()

In [ ]:
# Timing comparison — total time per method across all files
timing = (
    df_comparison
    .groupby('method')['time_s']
    .sum()
    .rename('total_time_s')
    .reset_index()
    .sort_values('total_time_s')
)
timing['relative_to_fastest'] = (timing['total_time_s'] / timing['total_time_s'].min()).round(1)

print("=== Total processing time across all files ===")
display(timing)